# Zolai — Qwen2.5 LoRA fine-tuning (Kaggle)

*Notebook: `zolai-qwen-training-v2.ipynb`*

## Run order (top → bottom)

| Step | Cell | Purpose |
|------|------|---------|
| 1 | Install | `pip` + env vars + versions / GPU probe |
| 2 | Config + load | Paths, copy dataset to `/kaggle/working`, splits |
| 3 | HF login | `HF_TOKEN` (Secrets or env) |
| 4 | Prepare | Filter / subsample `text` |
| 5 | Tokenizer | `AutoTokenizer` + pad token |
| 6 | Model | 4-bit Qwen + `prepare_model_for_kbit_training` (skipped when `USE_DDP_2GPU`; workers load per GPU) |
| 7 | Train | LoRA + `SFTTrainer` (`notebook_launcher` DDP when `USE_DDP_2GPU`) |
| 8 | Zip | Checkpoint zips (under `checkpoint_zips/`) + final adapter zip for download |

**Always use “Run All”** or run in this order. If you change `USE_DDP_2GPU` / `CUDA_VISIBLE_DEVICES`, **restart the session** and run from cell 1 again.

## Kaggle checklist

- **Settings → Accelerator:** GPU (e.g. T4 ×2)  
- **Internet:** on (model + tokenizer from Hub)  
- **Add data:** **peterpausianlian/zolai-hf-advanced** → `/kaggle/input/datasets/peterpausianlian/zolai-hf-advanced/` (or `ZOLOAI_DATASET_SRC`)  
- **Secrets:** `HF_TOKEN` for gated models  
- **Draft vs Save:** saving the notebook does not save outputs; **commit/version** if you need a stable snapshot  

High **CPU** and **0% GPU** during dataset copy or TRL preprocessing is normal; GPU use shows up in the **model** and **train** steps.

## Dataset: `zolai-hf-advanced`

This notebook expects a **`text`** column (plain language-modeling format). The **zolai-hf-advanced** dataset is **fine to train on as-is** if the text is already the content you want the model to imitate (same language/domain as production).

**Optional cleaning** (see prepare cell flags): turn on if you see broken encoding, noisy HTML, or excessive whitespace. **Do not** over-filter unless you have a clear issue—removing too many rows hurts diversity.

**When to preprocess offline instead:** wrong schema (e.g. only `instruction`/`response`), heavy deduplication across millions of rows, or PII scrubbing—fix those in a separate script or Hub dataset revision, then point `ZOLOAI_DATASET_SRC` at the cleaned `load_from_disk` folder.

## Recommendations

- **Smoke test:** set `MAX_TRAIN_SAMPLES = 2048` and lower `TOTAL_MAX_STEPS` in the config cell to verify the pipeline before a long run.  
- **Steps vs dataset size:** `TOTAL_MAX_STEPS` counts **optimizer steps**, not one row per step. With ~2.1M train rows you do **not** automatically cover “all rows” at 5000 steps — coverage depends on effective batch size and how many times the dataloader revisits samples. **One training cell run stops at `TOTAL_MAX_STEPS`**; it does **not** loop 5000 → 5000 → … until 2.1M rows unless you schedule multiple runs with a higher cumulative `TOTAL_MAX_STEPS` or repeated resume.  
- **Train on all ~2.1M rows (one full epoch):** in the config cell set **`MAX_TRAIN_SAMPLES = None`**, **`TRAIN_BY_EPOCHS = True`**, **`NUM_TRAIN_EPOCHS = 1`**. With `per_device_train_batch_size=1` and `gradient_accumulation_steps=8`, one epoch is on the order of **~dataset_size / 8** optimizer steps (~260k for 2.1M rows) — plan for **many hours** and **checkpoint resume** across Kaggle sessions if needed.  
- **Long runs & resume:** set `TOTAL_MAX_STEPS` to your **cumulative** target (e.g. `20000` for one long session). Keep `RESUME_FROM_CHECKPOINT = True` so after a disconnect you re-run **train** and continue from the latest `checkpoint-*` under `OUTPUT_DIR`. After a run **finishes**, raise `TOTAL_MAX_STEPS` again (or point `RESUME_FROM_CHECKPOINT` at an unzipped checkpoint from `checkpoint_zips/` and set the total to **previous global step + chunk**). Each checkpoint save also creates a **zip** in `/kaggle/working/checkpoint_zips/` (toggle `ZIP_CHECKPOINTS_FOR_DOWNLOAD`) for download and reuse.  
- **Disk:** dataset copy + checkpoints + zip need several GiB. **`ZIP_CHECKPOINTS_FOR_DOWNLOAD`** also writes a `.zip` per save under `checkpoint_zips/` (roughly one copy of each checkpoint on disk until you delete zips). If space is tight, set it **`False`** and rely on `OUTPUT_DIR/checkpoint-*` only. If `/kaggle/working` is small (~20 GiB), keep **`FORCE_RECOPY_DATASET = False`** after the first copy, trim **`save_total_limit`**, or delete old `checkpoint-*` / zips.  
- **OOM:** reduce `MAX_LENGTH`, `per_device_train_batch_size`, or `TOTAL_MAX_STEPS`; keep `gradient_accumulation_steps` higher to preserve effective batch size.  
- **Both T4s:** set **`USE_DDP_2GPU = True`** in the config cell for **DistributedDataParallel** (one 4-bit replica per GPU via Accelerate `notebook_launcher`). Set **`False`** to pin a single GPU. Restart the session after changing this.  

## Speed & performance (see config cell toggles)

- **`ATTN_IMPLEMENTATION="sdpa"`:** PyTorch 2 attention is usually faster than `eager`; fall back to `eager` if you see errors. FlashAttention-2 (`flash_attention_2`) can be faster still but needs a separate install and compatible GPU/build.
- **`GRADIENT_CHECKPOINTING=False`:** faster steps and often **better throughput** if you do **not** run out of memory; keep `True` on T4 if you OOM.
- **`PACKING=True`:** packs multiple short texts into one block — often **higher samples/sec**; enable only after a successful run with `PACKING=False`.
- **`EVAL_STEPS`:** larger value = **less time in eval** (faster wall-clock). For max quality tracking, lower it.
- **`MAX_LENGTH`:** shorter sequences = **faster steps** and less VRAM.
- **`per_device_train_batch_size`:** increase (e.g. 2) if VRAM allows — better GPU utilization; adjust `gradient_accumulation_steps` to keep effective batch size.
- **`DATALOADER_NUM_WORKERS`:** `2` can speed input pipeline vs `0`; set `0` if workers hang on Kaggle.
- **`torch.backends.cudnn.benchmark`:** enabled in the model cell for CUDA (small win on fixed shapes).
- **Long runs:** fewer checkpoint saves (`save_steps`) reduces I/O pauses.

**Implementation notes:** read-only `/kaggle/input` copy (optional skip), auto path fallback for `zolai-hf-advanced`, single `MODEL_NAME`, validation split fallback, TRL `SFTConfig` + `processing_class` / legacy `tokenizer` kwarg.


In [ ]:
# =========================
# 1. INSTALL DEPENDENCIES + SESSION CHECK
# =========================
# Core stack: transformers + TRL (SFT) + PEFT (LoRA) + bitsandbytes (4-bit) + accelerate
# + kaggle (CLI upload/version)
!pip install -q -U "transformers>=4.45.0" "datasets>=2.19.0" "peft>=0.12.0" "accelerate>=0.34.0" "bitsandbytes>=0.43.0" "trl>=0.9.0" "kaggle>=1.6.0"

import importlib.metadata as _im
import os
import subprocess
import sys

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print("Python:", sys.version.split()[0])
print("--- GPU (expect a line per GPU on Kaggle) ---")
try:
    subprocess.run(["nvidia-smi", "-L"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found (OK on CPU-only setups).")

print("--- Package versions ---")
for _pkg in ("torch", "transformers", "datasets", "trl", "peft", "accelerate", "bitsandbytes"):
    try:
        print(f"  {_pkg}: {_im.version(_pkg)}")
    except _im.PackageNotFoundError:
        print(f"  {_pkg}: (not installed)")

try:
    import torch

    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device:", torch.cuda.get_device_name(0))
except Exception as _e:
    print("torch import note:", _e)


In [ ]:
# =========================
# CONFIG + LOAD DATASET (copy off read-only input)
# =========================
import os

# Before `datasets`/torch: on Kaggle, pin visible GPUs. USE_DDP_2GPU=True → both T4s for DDP (train cell).
USE_DDP_2GPU = True  # False = single GPU (CUDA_VISIBLE_DEVICES=0 on Kaggle); True = 0,1 + Accelerate DDP
if os.path.isdir("/kaggle"):
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1" if USE_DDP_2GPU else "0"

from datasets import DatasetDict, load_from_disk

# --- Model & training defaults (edit here) ---
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_LENGTH = 512
SEED = 42
# Set to an int (e.g. 5000) for a quick dry run; None = use full train split
MAX_TRAIN_SAMPLES = None
VAL_SPLIT_FRAC = 0.01  # used only if the dataset has no validation/test split
# Kaggle: if working copy already exists (e.g. draft restart), skip re-copy to save time & disk
FORCE_RECOPY_DATASET = False

# Parallel workers for `dataset.map` in the prepare cell (normalization). 1 = safest; 2–4 = faster if RAM allows.
MAP_NUM_PROC = 2

# --- Speed vs VRAM (edit) ---
# "sdpa" uses PyTorch 2 scaled-dot-product attention (usually faster than "eager"). Use "eager" if errors.
ATTN_IMPLEMENTATION = "sdpa"
# False = faster steps if you still fit in VRAM; True = saves memory (typical for QLoRA on T4)
GRADIENT_CHECKPOINTING = True
# True can improve throughput by packing short sequences (test after one good run)
PACKING = False
# Larger = less time in eval (faster wall-clock)
EVAL_STEPS = 400
# 2 often OK on Kaggle; use 0 if worker crashes
DATALOADER_NUM_WORKERS = 2
# Aligns sequence lengths; can help tensor-core efficiency; None to disable
PAD_TO_MULTIPLE_OF = 8
# Global batch ≈ per_device_batch × world_size × this (batch size is 1 here). DDP×2 → use 4 to stay near single-GPU×8.
GRADIENT_ACCUMULATION_STEPS = 4 if USE_DDP_2GPU else 8

# --- Training schedule ---
# Total optimizer steps (global step target). One `trainer.train()` runs until this count is reached (then stops).
# Not the same as dataset rows (~2.1M): one step != one row; the notebook does NOT auto-chain another 5000.
# For chunked runs (e.g. 5000 then 5000 more): set `TOTAL_MAX_STEPS` to the cumulative target (10000) or resume
# from an unzipped checkpoint and set `TOTAL_MAX_STEPS` to last_step + chunk_steps.
TOTAL_MAX_STEPS = 5000
# Full dataset pass: set True with `MAX_TRAIN_SAMPLES = None` to train for `NUM_TRAIN_EPOCHS` over **all** train rows
# (e.g. one epoch ≈ 2.1M examples seen once each, modulo shuffle). Very long on GPU — use checkpoints + resume.
TRAIN_BY_EPOCHS = False
NUM_TRAIN_EPOCHS = 1
# Resume: None = start at step 0; True = latest checkpoint under OUTPUT_DIR (if any); or a path like OUTPUT_DIR + "/checkpoint-1000"
RESUME_FROM_CHECKPOINT = True

# --- Paths: on Kaggle this dataset resolves as below (Add data → peterpausianlian/zolai-hf-advanced) ---
KAGGLE_INPUT_DATASET = "/kaggle/input/datasets/peterpausianlian/zolai-hf-advanced"
KAGGLE_INPUT_DATASET_ALT = "/kaggle/input/zolai-hf-advanced"  # only if your mount uses the short slug
KAGGLE_WORKING_DATASET = "/kaggle/working/zolai_dataset"
KAGGLE_OUTPUT_DIR = "/kaggle/working/qwen-zolai"
KAGGLE_FINAL_ADAPTER = "/kaggle/working/qwen-zolai-final"
KAGGLE_ZIP_PREFIX = "/kaggle/working/qwen_zolai_lora"

# --- Optional uploads (edit here, then re-run final Zip cell) ---
# Kaggle dataset version upload (requires you own/collaborate on the dataset)
DO_KAGGLE_UPLOAD = False
KAGGLE_DATASET_ID = "peterpausianlian/zolai-qwen-training-v2"  # owner/slug
# IMPORTANT: dataset title must be unique per Kaggle account; notebook titles can also conflict.
# Use a distinct title for dataset outputs.
KAGGLE_DATASET_TITLE = "Zolai Qwen Training v2 - outputs"
# What to include in the Kaggle dataset version
KAGGLE_UPLOAD_INCLUDE_FINAL_ZIP = True
KAGGLE_UPLOAD_INCLUDE_CHECKPOINT_ZIPS = True  # includes files under CHECKPOINT_ZIP_DIR (if any)

# Kaggle *Model* upload (newer "Models" registry: model + instance/variation)
# This is separate from Kaggle Datasets. Enable if you want `kaggle models ...` upload.
DO_KAGGLE_MODEL_UPLOAD = False
# Model registry owner/slug, e.g. "peterpausianlian/zolai-qwen2.5-3b-lora" (Kaggle model registry, not HF)
KAGGLE_MODEL_ID = "peterpausianlian/zolai-qwen2-5-3b-lora"
KAGGLE_MODEL_TITLE = "Zolai Qwen2.5-3B LoRA"
# Instance/variation slug for this run (must be unique per model)
KAGGLE_MODEL_INSTANCE_SLUG = "lora-latest"
# Some Kaggle model-instance templates require framework fields
KAGGLE_MODEL_FRAMEWORK = "pytorch"
# Include files
KAGGLE_MODEL_INCLUDE_FINAL_ZIP = True
KAGGLE_MODEL_INCLUDE_CHECKPOINT_ZIPS = False

# Hugging Face Hub upload (uploads FINAL_ADAPTER_DIR folder to an existing model repo)
DO_HF_UPLOAD = False
HF_MODEL_REPO_ID = "peterpausianlian/zolai-qwen2.5-3b-lora"  # existing model repo id
HF_PATH_IN_REPO = "adapters/latest"  # where to place adapter files inside the repo
HF_UPLOAD_ZIP_TOO = True  # also upload qwen_zolai_lora.zip to repo root

IS_KAGGLE = os.path.isdir("/kaggle")
if IS_KAGGLE:
    WORKING_DATASET_PATH = KAGGLE_WORKING_DATASET
    OUTPUT_DIR = KAGGLE_OUTPUT_DIR
    FINAL_ADAPTER_DIR = KAGGLE_FINAL_ADAPTER
    ZIP_PREFIX = KAGGLE_ZIP_PREFIX
    WORK_DIR = "/kaggle/working"
    os.makedirs(WORK_DIR, exist_ok=True)
else:
    WORK_DIR = os.environ.get("ZOLOAI_WORK_DIR", ".")
    WORKING_DATASET_PATH = os.path.join(WORK_DIR, "zolai_dataset")
    OUTPUT_DIR = os.path.join(WORK_DIR, "qwen-zolai")
    FINAL_ADAPTER_DIR = os.path.join(WORK_DIR, "qwen-zolai-final")
    ZIP_PREFIX = os.path.join(WORK_DIR, "qwen_zolai_lora")
    os.makedirs(WORK_DIR, exist_ok=True)

# Zip each HF Trainer checkpoint here for Kaggle download (extra disk; set False if tight on space)
CHECKPOINT_ZIP_DIR = os.path.join(WORK_DIR, "checkpoint_zips")
ZIP_CHECKPOINTS_FOR_DOWNLOAD = True
os.makedirs(CHECKPOINT_ZIP_DIR, exist_ok=True)


def _is_hf_disk_root(path: str) -> bool:
    """True if `path` looks like a datasets.save_to_disk / DatasetDict folder."""
    if not path or not os.path.isdir(path):
        return False
    if os.path.isfile(os.path.join(path, "dataset_dict.json")):
        return True
    if os.path.isfile(os.path.join(path, "dataset_info.json")):
        return True
    return os.path.isfile(os.path.join(path, "train", "dataset_info.json"))


def _resolve_dataset_src() -> str:
    override = os.environ.get("ZOLOAI_DATASET_SRC")
    if override and _is_hf_disk_root(override):
        return os.path.abspath(override)
    if IS_KAGGLE:
        for p in (KAGGLE_INPUT_DATASET, KAGGLE_INPUT_DATASET_ALT):
            if _is_hf_disk_root(p):
                print("Using dataset:", p)
                return p
        inp = "/kaggle/input"
        if os.path.isdir(inp):
            zolai, other = [], []
            for name in sorted(os.listdir(inp)):
                cand = os.path.join(inp, name)
                if not _is_hf_disk_root(cand):
                    continue
                (zolai if "zolai" in name.lower() else other).append(cand)
            if zolai:
                print("Using dataset (name match):", zolai[0])
                return zolai[0]
            if other:
                print("Using dataset (first HF dataset in /kaggle/input):", other[0])
                return other[0]
    local = os.path.join(WORK_DIR, "zolai-hf-advanced")
    if _is_hf_disk_root(local):
        return local
    if override:
        return os.path.abspath(override)
    return local


DATASET_SRC = _resolve_dataset_src()

if not _is_hf_disk_root(DATASET_SRC):
    raise FileNotFoundError(
        f"Not a Hugging Face `load_from_disk` dataset folder: {DATASET_SRC!r}. "
        "On Kaggle add dataset **peterpausianlian/zolai-hf-advanced** (path `.../datasets/peterpausianlian/zolai-hf-advanced`), "
        "or set env `ZOLOAI_DATASET_SRC` to the folder "
        "that contains `dataset_dict.json` or `train/dataset_info.json`."
    )


def _working_copy_exists(path: str) -> bool:
    """True if a previous save_to_disk / DatasetDict layout is present."""
    if not os.path.isdir(path):
        return False
    return os.path.isfile(os.path.join(path, "dataset_dict.json")) or os.path.isfile(
        os.path.join(path, "dataset_info.json")
    )


# Copy off read-only Kaggle input; locally read straight from DATASET_SRC
if IS_KAGGLE:
    if (not FORCE_RECOPY_DATASET) and _working_copy_exists(WORKING_DATASET_PATH):
        print("Reusing existing working copy:", WORKING_DATASET_PATH)
        load_path = WORKING_DATASET_PATH
    else:
        print("Copying dataset to", KAGGLE_WORKING_DATASET, "…")
        _ds = load_from_disk(DATASET_SRC)
        _ds.save_to_disk(WORKING_DATASET_PATH)
        load_path = WORKING_DATASET_PATH
else:
    load_path = DATASET_SRC

raw = load_from_disk(load_path)
if not isinstance(raw, DatasetDict):
    raw = DatasetDict({"train": raw})

# Train / eval splits
train_ds = raw["train"]
if "validation" in raw:
    val_ds = raw["validation"]
elif "valid" in raw:
    val_ds = raw["valid"]
elif "test" in raw:
    val_ds = raw["test"]
else:
    print("No validation split; holding out a slice of train for eval.")
    split = train_ds.train_test_split(test_size=VAL_SPLIT_FRAC, seed=SEED)
    train_ds, val_ds = split["train"], split["test"]

print("train:", len(train_ds), "| val:", len(val_ds))

if IS_KAGGLE:
    import shutil

    du = shutil.disk_usage("/kaggle/working")
    total_gib = du.total / (1024**3)
    used_gib = du.used / (1024**3)
    free_gib = du.free / (1024**3)
    print(
        f"Disk (/kaggle/working): {free_gib:.1f} GiB free, {used_gib:.1f} GiB used, "
        f"{total_gib:.1f} GiB total volume"
    )
    if total_gib < 25:
        print(
            "Note: working volume is small — LoRA checkpoints + final adapter + zip need space. "
            "If training fails with “No space left”, raise `save_steps`, lower `save_total_limit`, "
            "or delete `/kaggle/working/qwen-zolai/checkpoint-*` between runs."
        )
    if free_gib < 5.0:
        print("Warning: critically low free space — training may fail mid-run.")
    elif free_gib < 12.0:
        print(
            "Warning: limited headroom for checkpoints. Use FORCE_RECOPY_DATASET=False after the first copy, "
            "and remove old `qwen_zolai_lora.zip` / checkpoints if needed."
        )


In [ ]:
# =========================
# HUGGING FACE LOGIN (gated models / tokenizer)
# =========================
import os

from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

if not HF_TOKEN:
    raise RuntimeError(
        "Set HF_TOKEN: Kaggle → Add-ons → Secrets → HF_TOKEN, or export HF_TOKEN locally."
    )

login(token=HF_TOKEN, add_to_git_credential=False)

try:
    from huggingface_hub import whoami

    _u = whoami()
    _name = _u.get("name") or _u.get("fullname") or _u.get("preferred_username") or "(unknown)"
    print("HF login OK — Hub user:", _name)
except Exception as _e:
    print("login() finished; could not verify via whoami:", _e)


### Prepare step (can take a few minutes)

The next cell runs **`dataset.map`** over **all train rows** when `NORMALIZE_TEXT` or `MIN_TEXT_CHARS > 1` is active. On ~2M examples you will see a **Map** progress bar (e.g. 20–90s per million rows depending on CPU). To iterate faster, set **`MAX_TRAIN_SAMPLES`** in the config cell above, or raise **`MAP_NUM_PROC`** (e.g. `4`) if you have RAM headroom; use **`MAP_NUM_PROC = 1`** if the kernel crashes or hangs.

In [ ]:
# =========================
# PREPARE DATASET (keep `text` for SFTTrainer)
# =========================
import re
import unicodedata

# zolai-hf-advanced: usually fine as-is; tune only if you see encoding/whitespace issues
NORMALIZE_TEXT = True  # NFKC + collapse whitespace
MIN_TEXT_CHARS = 1  # raise (e.g. 20) to drop very short lines

if "text" not in train_ds.column_names:
    raise KeyError(
        "Expected a `text` column. Map your field to `text` first if the schema differs."
    )

if MAX_TRAIN_SAMPLES is not None:
    n = min(MAX_TRAIN_SAMPLES, len(train_ds))
    train_ds = train_ds.shuffle(seed=SEED).select(range(n))


def _clean_one(text):
    t = str(text).strip()
    if NORMALIZE_TEXT:
        t = unicodedata.normalize("NFKC", t)
        t = re.sub(r"\s+", " ", t).strip()
    return t


if NORMALIZE_TEXT or MIN_TEXT_CHARS > 1:
    _np = max(1, int(MAP_NUM_PROC))
    train_ds = train_ds.map(
        lambda x: {"text": _clean_one(x["text"])},
        num_proc=_np,
    )
    val_ds = val_ds.map(
        lambda x: {"text": _clean_one(x["text"])},
        num_proc=min(_np, 4),
    )

train_ds = train_ds.filter(
    lambda x: bool(x.get("text")) and len(str(x["text"]).strip()) >= MIN_TEXT_CHARS,
    num_proc=1,
)
val_ds = val_ds.filter(
    lambda x: bool(x.get("text")) and len(str(x["text"]).strip()) >= MIN_TEXT_CHARS,
    num_proc=1,
)

print("After prep — train:", len(train_ds), "| val:", len(val_ds))
if len(train_ds) == 0 or len(val_ds) == 0:
    raise RuntimeError("train or eval split is empty after filtering; check your dataset `text` column.")


In [ ]:
# =========================
# TOKENIZER (same MODEL_NAME as training)
# =========================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
# =========================
# MODEL (4-bit QLoRA base)
# =========================
import torch
from peft import prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

if globals().get("USE_DDP_2GPU", False):
    print(
        "USE_DDP_2GPU=True: skipping model load in this cell — each DDP worker loads a full 4-bit replica on its GPU."
    )
    model = None
else:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    _attn_kw = {}
    if ATTN_IMPLEMENTATION:
        _attn_kw["attn_implementation"] = ATTN_IMPLEMENTATION
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            dtype=torch.float16,
            token=HF_TOKEN,
            trust_remote_code=True,
            **_attn_kw,
        )
    except (TypeError, ValueError) as _e:
        print("attn_implementation fallback (eager):", _e)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            dtype=torch.float16,
            token=HF_TOKEN,
            trust_remote_code=True,
        )

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True

    model.config.use_cache = False
    # Do not set torch_dtype on config (deprecated in Transformers 5); dtype is set via from_pretrained(dtype=...).
    if hasattr(model.config, "dtype"):
        model.config.dtype = torch.float16

    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING
    )

    for _, param in model.named_parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)

    print("CUDA available:", torch.cuda.is_available(), "| devices visible:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for _i in range(torch.cuda.device_count()):
            print(f"GPU{_i}:", torch.cuda.get_device_name(_i))


In [ ]:
# =========================
# LORA + SFT (TRL tokenizes `text`; no manual map)
# =========================
import os
import shutil

import torch
from peft import LoraConfig
from transformers import TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer


class ZipCheckpointCallback(TrainerCallback):
    """After each save, zip `checkpoint-*` into CHECKPOINT_ZIP_DIR for Kaggle download / offline resume."""

    def __init__(self, zip_dir: str):
        self.zip_dir = zip_dir

    def on_save(self, args, state, control, **kwargs):
        try:
            import torch.distributed as dist

            if dist.is_available() and dist.is_initialized() and dist.get_rank() != 0:
                return control
        except Exception:
            pass
        ckpt = get_last_checkpoint(args.output_dir)
        if not ckpt or not os.path.isdir(ckpt):
            return control
        base = os.path.basename(ckpt.rstrip(os.sep))
        out_base = os.path.join(self.zip_dir, base)
        try:
            shutil.make_archive(out_base, "zip", root_dir=ckpt)
            print(f"[checkpoint zip] {out_base}.zip")
        except OSError as e:
            print(f"[checkpoint zip] skipped: {e}")
        return control


def _ddp_train(cfg: dict) -> None:
    """Runs on each DDP process; loads 4-bit model on LOCAL_RANK."""
    import os
    import shutil

    import torch
    from datasets import load_from_disk
    from peft import LoraConfig, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from transformers.trainer_utils import get_last_checkpoint
    from trl import SFTConfig, SFTTrainer

    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)

    train_ds = load_from_disk(cfg["prep_train"])
    val_ds = load_from_disk(cfg["prep_val"])
    tokenizer = AutoTokenizer.from_pretrained(
        cfg["model_name"], token=cfg["hf_token"], trust_remote_code=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    _attn_kw = {}
    if cfg.get("attn_implementation"):
        _attn_kw["attn_implementation"] = cfg["attn_implementation"]
    try:
        model = AutoModelForCausalLM.from_pretrained(
            cfg["model_name"],
            quantization_config=bnb_config,
            device_map={"": local_rank},
            dtype=torch.float16,
            token=cfg["hf_token"],
            trust_remote_code=True,
            **_attn_kw,
        )
    except (TypeError, ValueError) as _e:
        if local_rank == 0:
            print("attn_implementation fallback (eager):", _e)
        model = AutoModelForCausalLM.from_pretrained(
            cfg["model_name"],
            quantization_config=bnb_config,
            device_map={"": local_rank},
            dtype=torch.float16,
            token=cfg["hf_token"],
            trust_remote_code=True,
        )

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
    model.config.use_cache = False
    if hasattr(model.config, "dtype"):
        model.config.dtype = torch.float16
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=cfg["gradient_checkpointing"]
    )
    for _, param in model.named_parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    _use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    _sft_kw = dict(
        output_dir=cfg["output_dir"],
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
        logging_steps=50,
        save_steps=200,
        save_total_limit=3,
        logging_first_step=True,
        eval_strategy="steps",
        eval_steps=cfg["eval_steps"],
        load_best_model_at_end=False,
        learning_rate=2e-4,
        fp16=not _use_bf16,
        bf16=_use_bf16,
        optim="paged_adamw_32bit",
        report_to="none",
        max_length=cfg["max_length"],
        dataset_text_field="text",
        shuffle_dataset=True,
        dataset_num_proc=1,
        dataloader_num_workers=cfg["dataloader_num_workers"],
        remove_unused_columns=False,
        gradient_checkpointing=cfg["gradient_checkpointing"],
        packing=cfg["packing"],
        ddp_find_unused_parameters=False,
    )
    if cfg["train_by_epochs"]:
        _sft_kw["max_steps"] = -1
        _sft_kw["num_train_epochs"] = float(cfg["num_train_epochs"])
    else:
        _sft_kw["max_steps"] = cfg["total_max_steps"]
    if cfg.get("pad_to_multiple_of") is not None:
        _sft_kw["pad_to_multiple_of"] = cfg["pad_to_multiple_of"]
    if not _use_bf16:
        _sft_kw["max_grad_norm"] = 0.0

    sft_config = SFTConfig(**_sft_kw)

    _callbacks = []
    if cfg.get("zip_checkpoints") and cfg.get("zip_checkpoint_dir"):
        os.makedirs(cfg["zip_checkpoint_dir"], exist_ok=True)
        _callbacks.append(ZipCheckpointCallback(cfg["zip_checkpoint_dir"]))

    _common_kw = dict(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        peft_config=lora_config,
        callbacks=_callbacks,
    )
    try:
        trainer = SFTTrainer(**_common_kw, processing_class=tokenizer)
    except TypeError:
        trainer = SFTTrainer(**_common_kw, tokenizer=tokenizer)

    if local_rank == 0:
        os.makedirs(cfg["final_adapter_dir"], exist_ok=True)
        print(
            "Starting DDP training…",
            "| train rows:",
            len(train_ds),
            "| val rows:",
            len(val_ds),
            "| packing:",
            cfg["packing"],
            "| gc:",
            cfg["gradient_checkpointing"],
        )
        print("Mixed precision:", "bf16" if _use_bf16 else "fp16 (grad clip disabled; QLoRA+scaler workaround)")

    _resume = None
    rfc = cfg["resume_from_checkpoint"]
    if rfc is True:
        _resume = get_last_checkpoint(cfg["output_dir"])
    elif isinstance(rfc, str) and rfc.strip():
        _rp = rfc.strip()
        _resume = _rp if os.path.isdir(_rp) else None
    if local_rank == 0:
        print("resume_from_checkpoint:", _resume or "(none — start from step 0)")
        if cfg["train_by_epochs"]:
            print("schedule: epochs =", float(cfg["num_train_epochs"]), "| max_steps = -1 (epoch-based)")
        else:
            print("schedule: max_steps (total target) =", cfg["total_max_steps"])

    trainer.train(resume_from_checkpoint=_resume)

    if local_rank == 0:
        trainer.model.save_pretrained(cfg["final_adapter_dir"])
        tokenizer.save_pretrained(cfg["final_adapter_dir"])
        print(f"Adapter + tokenizer saved to {cfg['final_adapter_dir']}")


_USE_DDP = globals().get("USE_DDP_2GPU", False)

if _USE_DDP:
    _prep_root = os.path.join(WORK_DIR, "sft_prepared")
    os.makedirs(_prep_root, exist_ok=True)
    _pt = os.path.join(_prep_root, "train")
    _pv = os.path.join(_prep_root, "val")
    print("DDP: saving prepared train/val to disk for worker processes…")
    train_ds.save_to_disk(_pt)
    val_ds.save_to_disk(_pv)

    _ddp_cfg = {
        "model_name": MODEL_NAME,
        "output_dir": OUTPUT_DIR,
        "final_adapter_dir": FINAL_ADAPTER_DIR,
        "prep_train": _pt,
        "prep_val": _pv,
        "max_length": MAX_LENGTH,
        "total_max_steps": TOTAL_MAX_STEPS,
        "train_by_epochs": globals().get("TRAIN_BY_EPOCHS", False),
        "num_train_epochs": float(globals().get("NUM_TRAIN_EPOCHS", 1)),
        "eval_steps": EVAL_STEPS,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "packing": PACKING,
        "pad_to_multiple_of": PAD_TO_MULTIPLE_OF,
        "dataloader_num_workers": DATALOADER_NUM_WORKERS,
        "attn_implementation": ATTN_IMPLEMENTATION,
        "hf_token": HF_TOKEN,
        "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
        "zip_checkpoint_dir": CHECKPOINT_ZIP_DIR,
        "zip_checkpoints": ZIP_CHECKPOINTS_FOR_DOWNLOAD,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    }

    from accelerate import notebook_launcher

    notebook_launcher(_ddp_train, args=(_ddp_cfg,), num_processes=2)
else:
    if model is None:
        raise RuntimeError(
            "Model is not loaded. If USE_DDP_2GPU is False, run the model cell above first."
        )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    # QLoRA + FP16 GradScaler on torch 2.1+ often breaks (bf16 vs fp16 grad unscale). Prefer BF16 when CUDA supports it.
    _use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    _sft_kw = dict(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        logging_steps=50,
        save_steps=200,
        save_total_limit=3,
        logging_first_step=True,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        load_best_model_at_end=False,
        learning_rate=2e-4,
        fp16=not _use_bf16,
        bf16=_use_bf16,
        optim="paged_adamw_32bit",
        report_to="none",
        max_length=MAX_LENGTH,
        dataset_text_field="text",
        shuffle_dataset=True,
        dataset_num_proc=1,
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        remove_unused_columns=False,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        packing=PACKING,
    )
    if globals().get("TRAIN_BY_EPOCHS", False):
        _sft_kw["max_steps"] = -1
        _sft_kw["num_train_epochs"] = float(globals().get("NUM_TRAIN_EPOCHS", 1))
    else:
        _sft_kw["max_steps"] = TOTAL_MAX_STEPS
    if PAD_TO_MULTIPLE_OF is not None:
        _sft_kw["pad_to_multiple_of"] = PAD_TO_MULTIPLE_OF

    if not _use_bf16:
        _sft_kw["max_grad_norm"] = 0.0

    sft_config = SFTConfig(**_sft_kw)

    _callbacks = []
    _zip_dir = globals().get("CHECKPOINT_ZIP_DIR")
    _zip_on = globals().get("ZIP_CHECKPOINTS_FOR_DOWNLOAD", True)
    if _zip_on and _zip_dir:
        os.makedirs(_zip_dir, exist_ok=True)
        _callbacks.append(ZipCheckpointCallback(_zip_dir))

    _common_kw = dict(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        peft_config=lora_config,
        callbacks=_callbacks,
    )
    try:
        trainer = SFTTrainer(**_common_kw, processing_class=tokenizer)
    except TypeError:
        trainer = SFTTrainer(**_common_kw, tokenizer=tokenizer)

    os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
    print(
        "Starting training…",
        "| train rows:",
        len(train_ds),
        "| val rows:",
        len(val_ds),
        "| packing:",
        PACKING,
        "| gc:",
        GRADIENT_CHECKPOINTING,
    )
    print("Mixed precision:", "bf16" if _use_bf16 else "fp16 (grad clip disabled; QLoRA+scaler workaround)")

    _resume = None
    if RESUME_FROM_CHECKPOINT is True:
        _resume = get_last_checkpoint(OUTPUT_DIR)
    elif isinstance(RESUME_FROM_CHECKPOINT, str) and RESUME_FROM_CHECKPOINT.strip():
        _rp = RESUME_FROM_CHECKPOINT.strip()
        _resume = _rp if os.path.isdir(_rp) else None
    print("resume_from_checkpoint:", _resume or "(none — start from step 0)")
    if globals().get("TRAIN_BY_EPOCHS", False):
        print("schedule: epochs =", float(globals().get("NUM_TRAIN_EPOCHS", 1)), "| max_steps = -1 (epoch-based)")
    else:
        print("schedule: max_steps (total target) =", TOTAL_MAX_STEPS)

    trainer.train(resume_from_checkpoint=_resume)

    trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
    tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
    print(f"Adapter + tokenizer saved to {FINAL_ADAPTER_DIR}")


In [ ]:
# =========================
# ZIP FOR DOWNLOAD (Kaggle) + OPTIONAL UPLOADS (Kaggle Dataset + Hugging Face Hub)
# =========================
import glob
import json
import os
import shutil
import subprocess
import tempfile

from IPython.display import FileLink, display
from huggingface_hub import HfApi, whoami

# NOTE: you might run this cell in a fresh session without re-running the config cell.
# In that case, derive a sensible default so checkpoint zips still show up.
_cz = globals().get("CHECKPOINT_ZIP_DIR")
if not _cz:
    _work_dir = globals().get("WORK_DIR", "/kaggle/working" if os.path.isdir("/kaggle") else os.getcwd())
    _cz = os.path.join(_work_dir, "checkpoint_zips")

if _cz and os.path.isdir(_cz):
    zips = sorted(glob.glob(os.path.join(_cz, "*.zip")))
    if zips:
        print(
            "Checkpoint zips (resume: unzip under /kaggle/working, set RESUME_FROM_CHECKPOINT to the unzipped checkpoint-* folder):"
        )
        for z in zips:
            rel = os.path.relpath(z, os.getcwd())
            display(FileLink(rel, result_html_prefix="checkpoint: "))
    else:
        print("No checkpoint zips yet — they appear after each save if ZIP_CHECKPOINTS_FOR_DOWNLOAD is True.")
else:
    print("(checkpoint zips) none found.")

if not os.path.isdir(FINAL_ADAPTER_DIR):
    raise FileNotFoundError(
        f"Nothing to zip — run the training cell first. Expected adapter at: {FINAL_ADAPTER_DIR!r}"
    )
if not any(os.scandir(FINAL_ADAPTER_DIR)):
    raise FileNotFoundError(f"Adapter folder is empty: {FINAL_ADAPTER_DIR!r}")

zip_path = shutil.make_archive(ZIP_PREFIX, "zip", FINAL_ADAPTER_DIR)
zip_path = os.path.abspath(zip_path)
print(f"Zipping complete: {zip_path}")

# Link is relative to notebook working dir on Kaggle (/kaggle/working)
zip_name = os.path.basename(zip_path)
display(FileLink(zip_name, result_html_prefix="final adapter: "))

# -------------------------
# OPTIONAL: upload/version to a Kaggle dataset via CLI
# -------------------------
# Requires:
# - Kaggle notebook session, OR local machine with Kaggle API credentials configured
# - an existing dataset to version
#
# Set this to your dataset id (owner/slug). Default matches the notebook name.
KAGGLE_DATASET_ID = os.environ.get(
    "KAGGLE_DATASET_ID", globals().get("KAGGLE_DATASET_ID", "peterpausianlian/zolai-qwen-training-v2")
)
KAGGLE_DATASET_TITLE = os.environ.get(
    "KAGGLE_DATASET_TITLE", globals().get("KAGGLE_DATASET_TITLE", "Zolai Qwen Training v2 - outputs")
)

DO_KAGGLE_UPLOAD = bool(int(os.environ.get("DO_KAGGLE_UPLOAD", "1" if globals().get("DO_KAGGLE_UPLOAD") else "0")))

def _run_kaggle(cmd):
    res = subprocess.run(cmd, text=True, capture_output=True)
    out = (res.stdout or "").strip()
    err = (res.stderr or "").strip()
    if out:
        print(out)
    if err:
        print(err)
    combined = "\n".join([x for x in (out, err) if x])
    return res, combined


def _kaggle_ok(res, combined: str) -> bool:
    if getattr(res, "returncode", 1) != 0:
        return False
    bad_markers = [
        "Client Error:",
        "Dataset creation error:",
        "Error while trying to load upload info:",
        "Traceback",
    ]
    return not any(m in (combined or "") for m in bad_markers)


# Include toggles (from config cell or env overrides)
KAGGLE_UPLOAD_INCLUDE_FINAL_ZIP = bool(
    int(
        os.environ.get(
            "KAGGLE_UPLOAD_INCLUDE_FINAL_ZIP",
            "1" if globals().get("KAGGLE_UPLOAD_INCLUDE_FINAL_ZIP", True) else "0",
        )
    )
)
KAGGLE_UPLOAD_INCLUDE_CHECKPOINT_ZIPS = bool(
    int(
        os.environ.get(
            "KAGGLE_UPLOAD_INCLUDE_CHECKPOINT_ZIPS",
            "1" if globals().get("KAGGLE_UPLOAD_INCLUDE_CHECKPOINT_ZIPS", True) else "0",
        )
    )
)

if DO_KAGGLE_UPLOAD:
    with tempfile.TemporaryDirectory(prefix="kaggle_dataset_upload_") as tmp:
        included = []

        # Optionally include final adapter zip
        if KAGGLE_UPLOAD_INCLUDE_FINAL_ZIP:
            out_zip = os.path.join(tmp, os.path.basename(zip_path))
            shutil.copy2(zip_path, out_zip)
            included.append(os.path.basename(out_zip))

        # Optionally include checkpoint zips for later resume
        if KAGGLE_UPLOAD_INCLUDE_CHECKPOINT_ZIPS:
            if _cz and os.path.isdir(_cz):
                ckpt_zips = sorted(glob.glob(os.path.join(_cz, "*.zip")))
            else:
                ckpt_zips = []
            if ckpt_zips:
                ckpt_out_dir = os.path.join(tmp, "checkpoint_zips")
                os.makedirs(ckpt_out_dir, exist_ok=True)
                for p in ckpt_zips:
                    dst = os.path.join(ckpt_out_dir, os.path.basename(p))
                    shutil.copy2(p, dst)
                    included.append(os.path.join("checkpoint_zips", os.path.basename(dst)))

        if not included:
            raise RuntimeError(
                "DO_KAGGLE_UPLOAD=True but nothing selected to upload. "
                "Enable KAGGLE_UPLOAD_INCLUDE_FINAL_ZIP and/or KAGGLE_UPLOAD_INCLUDE_CHECKPOINT_ZIPS."
            )

        print("Kaggle upload will include:")
        for x in included:
            print(" -", x)

        # Initialize metadata (same approach used in many Kaggle notebook workflows)
        _init = ["kaggle", "datasets", "init", "-p", tmp]
        _run_kaggle(_init)

        def _kaggle_title_safe(title: str, suffix: str = "") -> str:
            # Kaggle datasets: title must be 6..50 chars.
            t = (title or "").strip() or "Zolai Qwen LoRA outputs"
            suf = (suffix or "").strip()
            # Ensure suffix fits
            if suf and len(suf) > 20:
                suf = suf[:20]
            # Fit within 50
            max_len = 50
            if suf:
                # keep at least 6 chars of base
                base_max = max(6, max_len - (len(suf) + 1))
                t = t[:base_max].rstrip()
                t = f"{t} {suf}".strip()
            # Enforce min length
            if len(t) < 6:
                t = (t + " outputs")[:6]
            return t[:50]

        # If the title collides (Kaggle can reject titles already used by notebooks), append a short timestamp + random suffix.
        if os.path.isdir("/kaggle"):
            import random
            import string
            import time

            rand = "".join(random.choice(string.ascii_lowercase + string.digits) for _ in range(4))
            suf = f"{time.strftime('%m%d-%H%M')}-{rand}"  # still short enough to fit 50
            KAGGLE_DATASET_TITLE = _kaggle_title_safe(KAGGLE_DATASET_TITLE, suffix=suf)
        else:
            KAGGLE_DATASET_TITLE = _kaggle_title_safe(KAGGLE_DATASET_TITLE)

        # Kaggle CLI expects dataset-metadata.json in the same folder
        meta = {
            "id": KAGGLE_DATASET_ID,
            "title": KAGGLE_DATASET_TITLE,
            "licenses": [{"name": "CC0-1.0"}],
        }
        with open(os.path.join(tmp, "dataset-metadata.json"), "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

        # Try version first (existing dataset). If that fails, try create (first-time dataset).
        version_cmd = [
            "kaggle",
            "datasets",
            "version",
            "-p",
            tmp,
            "-m",
            f"Upload trained LoRA + checkpoints ({os.path.basename(zip_path)})",
            "-r",
            "zip",
        ]
        print("Running:", " ".join(version_cmd))
        res, combined = _run_kaggle(version_cmd)
        if _kaggle_ok(res, combined):
            print("Kaggle dataset version created for:", KAGGLE_DATASET_ID)
        else:
            create_cmd = [
                "kaggle",
                "datasets",
                "create",
                "-p",
                tmp,
                "-r",
                "zip",
            ]
            print("Version failed; trying create:", " ".join(create_cmd))
            res2, combined2 = _run_kaggle(create_cmd)
            if _kaggle_ok(res2, combined2):
                print("Kaggle dataset created:", KAGGLE_DATASET_ID)
            else:
                print("Kaggle create/version did not succeed.")
                print(
                    "Common causes:\n"
                    "- **403 Forbidden**: you do not own/collaborate on `KAGGLE_DATASET_ID`.\n"
                    "- **Title already in use**: change `KAGGLE_DATASET_TITLE` (recommended: add timestamp).\n"
                    "- Kaggle CLI mismatch error like `KaggleObject... token`: re-run the install cell to upgrade `kaggle`.\n"
                )
                print("Continuing (will still attempt HF upload if enabled).")
else:
    print(
        "Upload step skipped. To upload to Kaggle, set `DO_KAGGLE_UPLOAD=True` in the config cell (or env var) and re-run this cell.\n"
        "This Kaggle dataset version can contain the trained LoRA zip and (optionally) checkpoint zips for later resume."
    )

# -------------------------
# OPTIONAL: upload to Kaggle *Models* registry (model + instance)
# -------------------------
# This uses the newer Kaggle CLI commands:
#   kaggle models create
#   kaggle models instances create
# and is separate from Kaggle Datasets.
DO_KAGGLE_MODEL_UPLOAD = bool(
    int(os.environ.get("DO_KAGGLE_MODEL_UPLOAD", "1" if globals().get("DO_KAGGLE_MODEL_UPLOAD") else "0"))
)
KAGGLE_MODEL_ID = os.environ.get("KAGGLE_MODEL_ID", globals().get("KAGGLE_MODEL_ID", "peterpausianlian/zolai-qwen2-5-3b-lora"))
KAGGLE_MODEL_TITLE = os.environ.get("KAGGLE_MODEL_TITLE", globals().get("KAGGLE_MODEL_TITLE", "Zolai Qwen2.5-3B LoRA"))
KAGGLE_MODEL_INSTANCE_SLUG = os.environ.get(
    "KAGGLE_MODEL_INSTANCE_SLUG", globals().get("KAGGLE_MODEL_INSTANCE_SLUG", "lora-latest")
)
KAGGLE_MODEL_INCLUDE_FINAL_ZIP = bool(
    int(
        os.environ.get(
            "KAGGLE_MODEL_INCLUDE_FINAL_ZIP",
            "1" if globals().get("KAGGLE_MODEL_INCLUDE_FINAL_ZIP", True) else "0",
        )
    )
)
KAGGLE_MODEL_INCLUDE_CHECKPOINT_ZIPS = bool(
    int(
        os.environ.get(
            "KAGGLE_MODEL_INCLUDE_CHECKPOINT_ZIPS",
            "1" if globals().get("KAGGLE_MODEL_INCLUDE_CHECKPOINT_ZIPS", False) else "0",
        )
    )
)
KAGGLE_MODEL_FRAMEWORK = os.environ.get(
    "KAGGLE_MODEL_FRAMEWORK", globals().get("KAGGLE_MODEL_FRAMEWORK", "pytorch")
)

if DO_KAGGLE_MODEL_UPLOAD:
    with tempfile.TemporaryDirectory(prefix="kaggle_model_upload_") as mdir:
        files = []
        if KAGGLE_MODEL_INCLUDE_FINAL_ZIP:
            dst = os.path.join(mdir, os.path.basename(zip_path))
            shutil.copy2(zip_path, dst)
            files.append(os.path.basename(dst))
        if KAGGLE_MODEL_INCLUDE_CHECKPOINT_ZIPS and _cz and os.path.isdir(_cz):
            ckpt_zips = sorted(glob.glob(os.path.join(_cz, "*.zip")))
            if ckpt_zips:
                outd = os.path.join(mdir, "checkpoint_zips")
                os.makedirs(outd, exist_ok=True)
                for p in ckpt_zips:
                    dst = os.path.join(outd, os.path.basename(p))
                    shutil.copy2(p, dst)
                    files.append(os.path.join("checkpoint_zips", os.path.basename(dst)))

        if not files:
            raise RuntimeError("DO_KAGGLE_MODEL_UPLOAD=True but no files selected to upload.")

        # Model metadata
        _run_kaggle(["kaggle", "models", "init", "-p", mdir])
        model_meta_path = os.path.join(mdir, "model-metadata.json")
        if os.path.isfile(model_meta_path):
            with open(model_meta_path, "r", encoding="utf-8") as f:
                mm = json.load(f)
        else:
            mm = {}
        owner, slug = KAGGLE_MODEL_ID.split("/", 1)
        mm.update({"id": KAGGLE_MODEL_ID, "title": KAGGLE_MODEL_TITLE, "slug": slug, "ownerSlug": owner})
        with open(model_meta_path, "w", encoding="utf-8") as f:
            json.dump(mm, f, indent=2)

        # Create model (may already exist)
        print("Creating/updating Kaggle Model:", KAGGLE_MODEL_ID)
        res_m, comb_m = _run_kaggle(["kaggle", "models", "create", "-p", mdir])
        if not _kaggle_ok(res_m, comb_m) and "already exists" not in (comb_m or "").lower():
            print("Kaggle model create may have failed; continuing to instance create.")

        # Instance metadata (variation)
        inst_dir = os.path.join(mdir, "instance")
        os.makedirs(inst_dir, exist_ok=True)
        # Put files inside instance dir as Kaggle expects
        for rel in files:
            src = os.path.join(mdir, rel)
            dst = os.path.join(inst_dir, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)

        _run_kaggle(["kaggle", "models", "instances", "init", "-p", inst_dir])
        inst_meta_path = os.path.join(inst_dir, "model-instance-metadata.json")
        if os.path.isfile(inst_meta_path):
            with open(inst_meta_path, "r", encoding="utf-8") as f:
                im = json.load(f)
        else:
            im = {}
        # Fill required/expected fields so Kaggle doesn't warn about default ownerSlug
        im.update(
            {
                "modelId": KAGGLE_MODEL_ID,
                "instanceSlug": KAGGLE_MODEL_INSTANCE_SLUG,
                "title": f"{KAGGLE_MODEL_TITLE} ({KAGGLE_MODEL_INSTANCE_SLUG})",
                "ownerSlug": owner,
                "modelSlug": slug,
            }
        )
        # Some templates use different keys; set them too if present/needed
        im.setdefault("slug", KAGGLE_MODEL_INSTANCE_SLUG)
        im.setdefault("framework", KAGGLE_MODEL_FRAMEWORK)
        with open(inst_meta_path, "w", encoding="utf-8") as f:
            json.dump(im, f, indent=2)

        print("Creating Kaggle Model instance:", KAGGLE_MODEL_ID, "/", KAGGLE_MODEL_INSTANCE_SLUG)
        _run_kaggle(["kaggle", "models", "instances", "create", "-p", inst_dir])
else:
    print(
        "Kaggle Model upload skipped. Enable `DO_KAGGLE_MODEL_UPLOAD=True` in the config cell to upload to Kaggle Models registry."
    )

# -------------------------
# OPTIONAL: upload/version to Hugging Face model repo
# -------------------------
# This uploads the *folder* you trained (`FINAL_ADAPTER_DIR`) to an existing Hub model repo.
# The repo can be private; permissions are based on the HF account tied to HF_TOKEN.
HF_MODEL_REPO_ID = os.environ.get(
    "HF_MODEL_REPO_ID", globals().get("HF_MODEL_REPO_ID", "peterpausianlian/zolai-qwen2.5-3b-lora")
)
HF_PATH_IN_REPO = os.environ.get("HF_PATH_IN_REPO", globals().get("HF_PATH_IN_REPO", "adapters/latest"))

DO_HF_UPLOAD = bool(int(os.environ.get("DO_HF_UPLOAD", "1" if globals().get("DO_HF_UPLOAD") else "0")))
HF_UPLOAD_ZIP_TOO = bool(
    int(os.environ.get("HF_UPLOAD_ZIP_TOO", "1" if globals().get("HF_UPLOAD_ZIP_TOO", True) else "0"))
)

if DO_HF_UPLOAD:
    try:
        _me = whoami()
        _name = _me.get("name") or _me.get("preferred_username") or "(unknown)"
        print("HF whoami:", _name)
    except Exception as e:
        print("HF whoami failed (login may still be OK):", e)

    api = HfApi()

    if not os.path.isdir(FINAL_ADAPTER_DIR) or not any(os.scandir(FINAL_ADAPTER_DIR)):
        raise FileNotFoundError(f"Adapter folder missing/empty: {FINAL_ADAPTER_DIR!r}")

    print("Uploading adapter folder to HF…", "repo:", HF_MODEL_REPO_ID, "path:", HF_PATH_IN_REPO)
    api.upload_folder(
        repo_id=HF_MODEL_REPO_ID,
        repo_type="model",
        folder_path=FINAL_ADAPTER_DIR,
        path_in_repo=HF_PATH_IN_REPO,
        commit_message=f"Update LoRA adapter ({os.path.basename(FINAL_ADAPTER_DIR)})",
    )
    print("HF upload complete (folder).")

    if HF_UPLOAD_ZIP_TOO and os.path.isfile(zip_path):
        zip_dest = os.environ.get("HF_ZIP_NAME", os.path.basename(zip_path))
        print("Uploading zip to HF…", "dest:", zip_dest)
        api.upload_file(
            repo_id=HF_MODEL_REPO_ID,
            repo_type="model",
            path_or_fileobj=zip_path,
            path_in_repo=zip_dest,
            commit_message=f"Add {zip_dest}",
        )
        print("HF upload complete (zip).")
else:
    print(
        "HF upload skipped. To upload to Hugging Face Hub, set `DO_HF_UPLOAD=True` in the config cell and re-run this cell."
    )


## Upload notes (Kaggle + HF)

### Kaggle **Datasets** upload (artifacts)

Kaggle enforces:
- **`dataset-metadata.json` required fields**: `id`, `title`, `licenses`
- **Title length**: **6–50 characters**
- Title can collide with notebook titles; prefer adding a short run suffix.

Recommended config (Cell 2):

```python
DO_KAGGLE_UPLOAD = True
KAGGLE_DATASET_ID = "peterpausianlian/zolai-qwen-training-v2"  # must be a dataset you own/collaborate on
KAGGLE_DATASET_TITLE = "Zolai Qwen2.5 LoRA outputs"  # keep short; notebook auto-appends MMDD-HHMM
KAGGLE_UPLOAD_INCLUDE_FINAL_ZIP = True
KAGGLE_UPLOAD_INCLUDE_CHECKPOINT_ZIPS = True
```

What gets uploaded:
- `qwen_zolai_lora.zip` (trained LoRA)
- `checkpoint_zips/*.zip` (optional resume checkpoints)

### Kaggle **Models** upload (model registry)

Enable if you want `kaggle models ...` + `kaggle models instances ...`:

```python
DO_KAGGLE_MODEL_UPLOAD = True
KAGGLE_MODEL_ID = "peterpausianlian/zolai-qwen2-5-3b-lora"
KAGGLE_MODEL_INSTANCE_SLUG = "lora-latest"  # or per-run slug like "lora-0330-1"
KAGGLE_MODEL_INCLUDE_FINAL_ZIP = True
```

### Hugging Face upload

```python
DO_HF_UPLOAD = True
HF_MODEL_REPO_ID = "peterpausianlian/zolai-qwen2.5-3b-lora"
HF_PATH_IN_REPO = "adapters/latest"
HF_UPLOAD_ZIP_TOO = True
```
